# T09 — 신호처리의 이론

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait-theory/note-T09.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait-theory/note-T09.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 본문

### 5. 필터: 진폭, 위상, 그리고 두 번 거르기

아래 코드는 이 두 사실 가운데 첫째를 확인하고, 다음 절의 미분 잡음 증폭을 숫자로 보인다.

In [ ]:
import numpy as np
from scipy import signal

fs, fc = 100.0, 6.0                       # 표본화 100 Hz, 원하는 cutoff 6 Hz
f = np.linspace(0.01, 20, 20000)

def gain_after_filtfilt(fc_design):
    b, a = signal.butter(2, fc_design / (fs / 2))
    _, h = signal.freqz(b, a, worN=f, fs=fs)
    return np.abs(h) ** 2                 # 앞뒤 두 번 지나가면 진폭 응답이 제곱된다

def minus3db(g):
    return f[np.argmin(np.abs(g - 1 / np.sqrt(2)))]

C = (2 ** 0.5 - 1) ** 0.25                # 2차 Butterworth, 두 번 통과 보정계수
fc_exact = fs / np.pi * np.arctan(np.tan(np.pi * fc / fs) / C)   # tan 보정까지 넣은 설계값
print(f"보정계수 C = {C:.3f}")
print(f"보정 없이 butter(2, 6 Hz)+filtfilt: 실제 -3 dB 지점 {minus3db(gain_after_filtfilt(fc)):.2f} Hz")
print(f"설계 cutoff를 {fc_exact:.2f} Hz로 올리면:       실제 -3 dB 지점 {minus3db(gain_after_filtfilt(fc_exact)):.2f} Hz")

# 미분이 잡음을 키우는 정도: 1 Hz 신호(진폭 50 mm) + 20 Hz 잡음(진폭 0.5 mm, 1%)
# n번째 harmonic의 진폭은 한 번 미분할 때마다 (2*pi*f)배 → 비율은 f의 k제곱으로 커진다
for k, name in [(1, "속도"), (2, "가속도")]:
    print(f"{name}: 잡음/신호 진폭비 = {(0.5 * 20**k) / (50 * 1**k):.0%}")